# UP2.3 · Pandas II: seleccionar y filtrar

Ejemplos de la presentación, en el mismo orden. Este notebook va en la raíz del proyecto, junto a la carpeta `datos/`.

Ejecuta las celdas de una en una, de arriba abajo. Al final tienes la referencia, el ejercicio *Ahora tú* y los errores que se pueden provocar a propósito.

Los datos son **simulados**: los barrios de Benidorm son reales, pero los anuncios y los precios se han generado con NumPy.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pisos: pd.DataFrame = pd.read_csv(Path("datos") / "pisos.csv")
columnas: list[str] = ["barrio", "tipo", "metros", "precio"]

## 1. Seleccionar con loc e iloc

In [2]:
print(pisos.loc[0, "precio"])
print(pisos.loc[0:2, ["barrio", "precio"]])

1830
     barrio  precio
0   Levante    1830
1   Foietes     570
2  Poniente    1040


In [3]:
print(pisos.iloc[0, 12])
print(pisos.iloc[0:2, [2, 12]])
print(pisos.iloc[-1, 2])

1830
    barrio  precio
0  Levante    1830
1  Foietes     570
Rincón de Loix


In [4]:
anuncios: pd.DataFrame = pisos.set_index("id")
print(anuncios.loc[1003, ["barrio", "precio"]])
print(anuncios.iloc[2]["precio"])

barrio    Poniente
precio        1040
Name: 1003, dtype: object
1040


### Referencia: loc e iloc

| Código | Qué selecciona |
|---|---|
| `df.loc[fila, columna]` | Por etiqueta: el índice y el nombre de la columna |
| `df.loc[0:2, ["a", "b"]]` | Filas con etiqueta de 0 a 2, **las dos incluidas** |
| `df.iloc[fila, columna]` | Por posición: 0 es la primera |
| `df.iloc[0:2, [2, 12]]` | Filas en las posiciones 0 y 1: **el fin no entra** |
| `df.iloc[-1]` | La última fila |
| `df.set_index("id")` | Usa una columna como índice (devuelve un DataFrame nuevo) |
| `df.reset_index(drop=True)` | Vuelve a numerar las filas desde 0 |

## 2. Filtrar filas

In [5]:
caros: pd.Series = pisos["precio"] > 3000
print(caros.sum())
print(pisos[caros].head(3))

7
       id       fecha          barrio  ... anunciante  precio  alquilado_30_dias
70   1071  2026-02-08         La Cala  ...    agencia    3620              False
219  1220  2026-04-15         Levante  ...    agencia    3930              False
230  1231  2026-04-23  Rincón de Loix  ...    agencia    3030              False

[3 rows x 14 columns]


In [6]:
print(pisos.loc[caros, ["barrio", "precio"]].head(3))

             barrio  precio
70          La Cala    3620
219         Levante    3930
230  Rincón de Loix    3030


In [7]:
print(pisos.precio.equals(pisos["precio"]))

pisos.precio_doble = pisos["precio"] * 2
print("precio_doble" in pisos.columns)

True


False


In [8]:
levante_mar: pd.DataFrame = pisos[
    (pisos["barrio"] == "Levante") & pisos["vistas_mar"]
]
print(len(levante_mar))

baratos_grandes: pd.Series = (
    (pisos["precio"] < 900) & (pisos["metros"] >= 100)
)
print(pisos.loc[baratos_grandes, columnas])

32
              barrio  tipo  metros  precio
405          Foietes  piso     101     840
592  Salt de l'Aigua  piso     115     870


In [9]:
poniente_cala: pd.Series = pisos["barrio"].isin(["Poniente", "La Cala"])
de_60_a_70: pd.Series = pisos["metros"].between(60, 70)
print(poniente_cala.sum(), de_60_a_70.sum())
print(pisos.loc[poniente_cala & de_60_a_70, columnas].head(3))

124 66
      barrio   tipo  metros  precio
2   Poniente   piso      70    1040
54  Poniente  ático      70    1240
75   La Cala   piso      63     990


In [10]:
print(pisos.query("precio > 3000 and vistas_mar")[columnas])

limite: int = 3500
print(pisos.query("precio > @limite")[columnas])

       barrio   tipo  metros  precio
256    Centro  ático     105    3120
472   Levante   piso      80    3110
511  Poniente  ático     124    3260
      barrio    tipo  metros  precio
70   La Cala  dúplex     126    3620
219  Levante  dúplex     108    3930


In [11]:
mas_caros: pd.DataFrame = pisos.loc[
    pisos["precio"] > 3000, ["barrio", "precio"]
]
print(mas_caros.iloc[0])
print(mas_caros.reset_index(drop=True).head(2))

barrio    La Cala
precio       3620
Name: 70, dtype: object
    barrio  precio
0  La Cala    3620
1  Levante    3930


### Referencia: filtrar filas

| Código | Qué hace |
|---|---|
| `df[df["precio"] > 3000]` | Filas que cumplen la condición |
| `(cond1) & (cond2)` · `\|` · `~` | Y · o · no, cada condición entre paréntesis |
| `df["barrio"].isin(["Poniente", "La Cala"])` | El valor está en la lista |
| `df["metros"].between(60, 70)` | Entre dos valores, los dos incluidos |
| `df.loc[condición, ["a", "b"]]` | Filas que cumplen y solo esas columnas, en un paso |

**Corchetes o `loc`:** corchetes para elegir solo columnas (`df["a"]`, `df[["a", "b"]]`) o solo filas (`df[condición]`); `loc` cuando son filas y columnas a la vez o cuando se cambian valores. `df[condición]["a"]` son dos pasos: el primero crea una tabla nueva, y por eso no sirve para asignar.

**El punto:** `df.precio` también lee una columna, pero no crea columnas (`df.nueva = ...` solo da un aviso) ni funciona con nombres como `size` o `count`, que ya son atributos del DataFrame. En este curso, siempre corchetes.
| `df.query("precio > 3000 and vistas_mar")` | La condición escrita como texto; admite `and`, `or`, `not` |
| `df.query("precio > @limite")` | Con `@` se usa una variable de Python |

## 3. Crear y modificar columnas

In [12]:
pisos["precio_m2"] = (pisos["precio"] / pisos["metros"]).round(2)
print(pisos[["metros", "precio", "precio_m2"]].head(3))

   metros  precio  precio_m2
0     115    1830      15.91
1      35     570      16.29
2      70    1040      14.86


In [13]:
pisos["gama"] = np.where(pisos["precio_m2"] > 18, "alta", "normal")
print(pisos["gama"].value_counts())

gama
normal    425
alta      175
Name: count, dtype: int64


In [14]:
print(pisos.loc[pisos["tipo"] == "estudio", "habitaciones"].unique())
pisos.loc[pisos["tipo"] == "estudio", "habitaciones"] = 0
print(pisos.loc[pisos["tipo"] == "estudio", "habitaciones"].unique())

[1]
[0]


In [15]:
pisos.rename(columns={"banos": "num_banos"})
print("banos" in pisos.columns)

pisos = pisos.rename(columns={"banos": "num_banos"})
print("num_banos" in pisos.columns)

sin_extras: pd.DataFrame = pisos.drop(columns=["precio_m2", "gama"])
print(pisos.shape, sin_extras.shape)

True
True
(600, 16) (600, 14)


### Referencia: crear y modificar columnas

| Código | Qué hace |
|---|---|
| `df["nueva"] = df["a"] / df["b"]` | Crea (o sustituye) una columna |
| `np.where(condición, "sí", "no")` | Un valor u otro según la condición |
| `df.loc[condición, "columna"] = valor` | Cambia solo las filas que cumplen |
| `df.rename(columns={"viejo": "nuevo"})` | Cambia nombres; **devuelve** un DataFrame nuevo |
| `df.drop(columns=["a", "b"])` | Quita columnas; **devuelve** un DataFrame nuevo |
| `df.assign(nueva=...)` | Añade una columna dentro de una cadena de métodos |

## 4. Ordenar, encadenar y copias

In [16]:
print(pisos.sort_values("precio", ascending=False)[columnas].head(3))
print(pisos.sort_values(["barrio", "precio"])[columnas].head(3))
print(pisos.nlargest(3, "precio_m2")[["barrio", "metros", "precio_m2"]])

       barrio    tipo  metros  precio
219   Levante  dúplex     108    3930
70    La Cala  dúplex     126    3620
511  Poniente   ático     124    3260
            barrio     tipo  metros  precio
157  Casco Antiguo  estudio      28     440
393  Casco Antiguo  estudio      31     540
470  Casco Antiguo     piso      53     570
      barrio  metros  precio_m2
472  Levante      80      38.88
197  Levante      39      36.92
219  Levante     108      36.39


In [17]:
mejores: pd.DataFrame = (
    pisos
    .query("vistas_mar and ascensor")
    .assign(
        precio_hab=lambda d: (d["precio"] / d["habitaciones"]).round()
    )
    .sort_values("precio_hab")
    .loc[:, ["barrio", "habitaciones", "precio", "precio_hab"]]
    .head(3)
)
print(mejores)

            barrio  habitaciones  precio  precio_hab
591        La Cala             3     880       293.0
213  Casco Antiguo             3     970       323.0
106        La Cala             4    1380       345.0


In [18]:
caros_copia: pd.DataFrame = pisos[pisos["precio"] > 3000]
caros_copia["precio"] = 0
print(pisos["precio"].max())

pisos[pisos["precio"] > 3000]["precio"] = 0
print(pisos["precio"].max())

3930


ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.


3930


### Referencia: ordenar y encadenar

| Código | Qué hace |
|---|---|
| `df.sort_values("precio")` | Ordena de menor a mayor |
| `df.sort_values("precio", ascending=False)` | De mayor a menor |
| `df.sort_values(["barrio", "precio"])` | Por barrio y, dentro de cada barrio, por precio |
| `df.nlargest(3, "precio")` · `df.nsmallest(3, "precio")` | Las 3 filas con el valor más alto · más bajo |
| `( df.query(...).assign(...).sort_values(...) )` | Cadena de métodos entre paréntesis, uno por línea |

**Copias en pandas 3:** el resultado de un filtro es siempre independiente del original. Para cambiar el original se usa `df.loc[condición, "columna"] = valor`, nunca `df[condición]["columna"] = valor`.

## Ahora tú

Responde a cada pregunta con una o dos líneas de pandas. Empieza volviendo a cargar el archivo, porque los ejemplos anteriores han modificado `pisos`.

1. ¿Cuántos pisos de Levante tienen vistas al mar? ¿Cuál es su precio medio?
2. Muestra el barrio, los metros y el precio de los cinco pisos más baratos que tienen ascensor y al menos dos habitaciones.
3. Crea una columna con el precio por metro cuadrado. ¿En qué barrio está el anuncio con el precio por metro más alto?
4. ¿Qué porcentaje de los áticos se alquiló en 30 días?

In [19]:
pisos = pd.read_csv(Path("datos") / "pisos.csv")

## Errores para provocar

Estas celdas dan error a propósito. Están al final para que *Ejecutar todo* no se detenga antes.

In [20]:
print(pisos[pisos["precio"] > 3000 and pisos["vistas_mar"]])

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [21]:
print(pisos[pisos["precio"] > 3000].loc[0])

KeyError: 0

In [22]:
pisos.loc[0, "precio"] = 1830.5

TypeError: Invalid value '1830.5' for dtype 'int64'